# Wheat detection - training (Part 7)
Before running, in the right-hand **Session options**: Accelerator = **GPU T4 x2**, Internet = **On**,
and **Add Input** -> the *Global Wheat Detection* competition.
Run once with `SMOKE = True` (a few minutes), then set it to `False` and use **Save Version -> Save & Run All**.
Download the results from the version's **Output** tab.

In [ ]:
REPO = "OWNER/wheat-mlops"   # <- your GitHub "owner/repo"
BRANCH = "training"          # <- branch that contains train.py and train_params.yaml
SMOKE = True                 # True = quick 2-epoch check, False = full training

In [ ]:
from pathlib import Path

Path("/kaggle/temp").mkdir(exist_ok=True)
REPO_DIR = Path("/kaggle/temp/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")  # only needed for a private repo
except Exception:
    pass
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"

In [ ]:
!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {url} {REPO_DIR}
!pip install -q "ultralytics>=8.3,<9"

In [ ]:
import json
import sys

import yaml

sys.path.insert(0, str(REPO_DIR / "src" / "python"))

try:
    from wheat_mlops.prepare_data import prepare
except ModuleNotFoundError as err:
    if err.name != "mlflow":
        raise
    print("prepare_data imports mlflow; installing the light version so the import works")
    !pip install -q mlflow-skinny
    from wheat_mlops.prepare_data import prepare


def find_raw() -> Path:
    for candidate in (
        "/kaggle/input/competitions/global-wheat-detection",
        "/kaggle/input/global-wheat-detection",
    ):
        if (Path(candidate) / "train.csv").is_file():
            return Path(candidate)
    hits = list(Path("/kaggle/input").rglob("train.csv"))
    assert hits, "Competition data not found - use Add Input -> Global Wheat Detection"
    return hits[0].parent


RAW = find_raw()
PROCESSED = Path("/kaggle/temp/wheat_processed")
p = yaml.safe_load((REPO_DIR / "params.yaml").read_text(encoding="utf-8"))["data"]
stats = prepare(
    raw_dir=RAW,
    out_dir=PROCESSED,
    val_fraction=p["val_fraction"],
    seed=p["seed"],
    count_bucket=p["count_bucket"],
    max_count_bucket=p["max_count_bucket"],
    min_box_area=p["min_box_area"],
    max_box_area=p["max_box_area"],
    link_images=False,  # /kaggle/input is a different filesystem, so copy
)
print(f"raw data: {RAW}")
print(json.dumps(stats, indent=2))

In [ ]:
FLAG = "--smoke" if SMOKE else ""
OUT_DIR = "/kaggle/temp/trained_smoke" if SMOKE else "/kaggle/working/trained"
!python {REPO_DIR}/src/python/wheat_mlops/train.py --data-yaml {PROCESSED}/dataset.yaml --out-dir {OUT_DIR} --params {REPO_DIR}/train_params.yaml {FLAG}

In [ ]:
print(Path(OUT_DIR, "metrics.json").read_text())
!ls -la {OUT_DIR}